In [ ]:
"""Evaluate the model"""
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, accuracy_score, auc,confusion_matrix,roc_curve
import pandas as pd
import matplotlib.pyplot as plt
import os

model_dir = os.getcwd()+ "/models/"
path = model_dir + 'ECG_Echo_CNN/'

calssificatin_type = "binary"   
no_of_classes =2

def sklearn_metrics(y_true, y_pred):
    y_bin = np.copy(y_pred)
    y_bin[y_bin >= 0.5] = 1
    y_bin[y_bin < 0.5]  = 0

    # Compute area under precision-Recall curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
    auc_sum = auc(recall, precision)

    TP = np.sum((y_true == 1) & (y_bin == 1))
    TN = np.sum((y_true == 0) & (y_bin == 0))
    FP = np.sum((y_true == 0) & (y_bin == 1))
    FN = np.sum((y_true == 1) & (y_bin == 0))

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    print("Accuracy        : {:.2f}".format(accuracy_score(y_true.flatten(), y_bin.flatten())* 100))
    print("Macro AUC score : {:.2f}".format(roc_auc_score(y_true, y_pred, average='macro') * 100))
    print('AUPRC           : {:.2f}'.format(auc_sum  * 100))
    print("Macro F1 score  : {:.2f}".format(f1_score(y_true, y_bin, average='macro') * 100))
    print("Sensitivity     : {:.2f}".format(sensitivity * 100))
    print("Specificity     : {:.2f}".format(specificity * 100))
    print("TP:", int(TP), "FP:", int(FP), "TN:", int(TN), "FN:", int(FN))


In [ ]:
y_train = np.load(path + 'y_train.npy',allow_pickle=True)
y_test = np.load(path + 'y_test.npy',allow_pickle=True)
y_val = np.load(path + 'y_val.npy',allow_pickle=True)
y_pred_train = np.load(path + 'y_pred_train.npy',allow_pickle=True)
y_pred_test = np.load(path + 'y_pred_test.npy',allow_pickle=True)
y_pred_val = np.load(path + 'y_pred_val.npy',allow_pickle=True)

print(y_train.shape)
print(y_pred_train.shape)
print(y_test.shape)
print(y_val.shape)


In [ ]:

print("Train")
sklearn_metrics(y_train.ravel(), y_pred_train.ravel())
print("\nTest")
sklearn_metrics(y_test.ravel(), y_pred_test.ravel())


In [ ]:

df = pd.DataFrame({
    "y_true": y_test.ravel(),
    "y_pred_prob": y_pred_test.ravel(),
    "y_pred_label": (y_pred_test.ravel() >= 0.5).astype(int)
})

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test.ravel(),(y_pred_test.ravel() >= 0.5).astype(int))
print(cm)


In [ ]:
# Compute precision-recall points
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_test)
# Compute area under the PR curve
auprc = auc(recall, precision)
print("AUPRC:", auprc)


plt.plot(recall, precision, label=f"AUC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend(loc="upper right")
plt.grid()
plt.show()


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], 'k--')  # diagonal for random classifier
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
def pick_threshold_rule_out(y_true, p, min_sensitivity=0.95):

    y_true = np.asarray(y_true).astype(int).ravel()
    p = np.asarray(p).ravel()

    fpr, tpr, thr = roc_curve(y_true, p)
    spec = 1 - fpr

    ok = tpr >= min_sensitivity
    if not np.any(ok):
        # If cannot reach the target sensitivity, take the threshold with best sensitivity.
        idx = np.argmax(tpr)
        return float(thr[idx]), float(tpr[idx]), float(spec[idx])

    # Among thresholds meeting sensitivity target, maximize specificity.
    idx_candidates = np.where(ok)[0]
    idx = idx_candidates[np.argmax(spec[ok])]
    return float(thr[idx]), float(tpr[idx]), float(spec[idx])

def eval_at_threshold(y_true, p, threshold):
    y_true = np.asarray(y_true).astype(int).ravel()
    p = np.asarray(p).ravel()
    y_hat = (p >= threshold).astype(int)
    precision, recall, thresholds = precision_recall_curve(y_true, p)
    auc_sum = auc(recall, precision)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    sens = tp / (tp + fn + 1e-12)
    spec = tn / (tn + fp + 1e-12)
    ppv  = tp / (tp + fp + 1e-12)
    npv  = tn / (tn + fn + 1e-12)
    return {
        "threshold": float(threshold),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Accuracy " : float(accuracy_score(y_true.flatten(), y_hat.flatten())),
        "AUPRC": float(auc_sum),
        "Macro AUC score":float(roc_auc_score(y_true, p, average='macro')),
        "Macro F1 score"  : float(f1_score(y_true, y_hat, average='macro')),
        "sensitivity": float(sens),
        "specificity": float(spec),
        "PPV": float(ppv),
        "NPV": float(npv),
    }
p_val = y_pred_val
t, sens, spec = pick_threshold_rule_out(y_val, p_val, min_sensitivity=0.95)
val_metrics = eval_at_threshold(y_val, p_val, t)
print("Chosen threshold:", t, "VAL sens:", sens, "VAL spec:", spec)
print(val_metrics)

In [ ]:
print('MIMIC train thr=0.5')
val_metrics = eval_at_threshold(y_train, y_pred_train, 0.5)
print(val_metrics)

print('MIMIC test thr=0.5')
val_metrics = eval_at_threshold(y_test, y_pred_test, 0.5)
print(val_metrics)

print('MIMIC val thr=0.5')
val_metrics = eval_at_threshold(y_val, y_pred_val, 0.5)
print(val_metrics)

print('MIMIC train thr =optimal')
val_metrics = eval_at_threshold(y_train, y_pred_train, t)
print(val_metrics)

print('MIMIC test thr=optimal')
val_metrics = eval_at_threshold(y_test, y_pred_test, t)
print(val_metrics)

print('MIMIC val thr=optimal')
val_metrics = eval_at_threshold(y_val, y_pred_val, t)
print(val_metrics)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
y_hat = (y_pred_test >= 0.5).astype(int)
correct = (y_hat == y_test)

y_pred_0 = y_pred_test[y_test == 0]
y_pred_1 = y_pred_test[y_test == 1]

plt.hist(y_pred_0, bins=50, alpha=0.6, label="y=0")
plt.hist(y_pred_1, bins=50, alpha=0.6, label="y=1")

plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.legend()
plt.title("Prediction distribution by true label")
plt.show()
